# ML Laboratuvarı · Modül 6, Ünite 9: Sterlin volatilitesi ve VaR

**Prof. Dr. Mehmet Songur** · ML Laboratuvarı, Modül 6

Veri: sterlin/dolar kuru, günlük, 1971'den bu yana (FRED: DEXUSUK, ABD Merkez Bankası H.10 serisi, kamu malı). Kurulum hücresi `kur` adlı serileri (tarih indeksli) indirir. Kodu **Gemini'ye yazdır**, sonra **denetle**.

**Kurallar**
1. **Kurulum** ve **Kontrol** hücrelerini değiştirme; yalnızca çalıştır.
2. Görev hücrelerinde kodu Gemini ile üret; çalıştırmadan önce oku.
3. Kontrol hücresi geçerse bir **tamamlama kodu** verir; onu ML Laboratuvarı'ndaki kutuya yapıştır.

In [ ]:
#@title Kurulum — çalıştır, değiştirme

import base64, warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score
from sklearn.metrics import r2_score
warnings.filterwarnings("ignore")

def _k(s):
    return base64.b64decode(s).decode()

def _referans(seri):
    r = 100 * np.log(seri).diff().dropna()
    t = pd.DataFrame(index=r.index)
    for w in [5, 20, 60]:
        t[f"v{w}"] = np.sqrt((r ** 2).rolling(w).mean())
    t["y_v"] = np.sqrt((r ** 2).rolling(20).mean().shift(-20))
    t = t.dropna()
    X, y = t[["v5", "v20", "v60"]], t["y_v"]
    e = t.index < "2015-01-01"
    out = dict(r=r, t=t,
               cv=cross_val_score(LinearRegression(), X, y, cv=TimeSeriesSplit(5, gap=20), scoring="r2").mean(),
               cv_nogap=cross_val_score(LinearRegression(), X, y, cv=TimeSeriesSplit(5), scoring="r2").mean(),
               cv_rand=cross_val_score(LinearRegression(), X, y, cv=KFold(5, shuffle=True, random_state=0), scoring="r2").mean(),
               test=r2_score(y[~e], LinearRegression().fit(X[e], y[e]).predict(X[~e])),
               naive=r2_score(y[~e], t.loc[~e, "v60"]))
    yr = r.shift(-1).reindex(t.index)
    oran = (yr / t["v60"]).dropna()
    ee = oran.index < "2015-01-01"
    q = np.quantile(oran[ee], 0.01)
    out.update(q=q, q_all=np.quantile(oran, 0.01), exc=(yr[~e] < q * t.loc[~e, "v60"]).mean(), exc_n=(yr[~e] < -2.326 * t.loc[~e, "v60"]).mean())
    return out

kur = pd.read_csv("https://raw.githubusercontent.com/mehmetsongur/ml-lab/main/data/fred/DEXUSUK.csv", index_col=0, parse_dates=True).iloc[:, 0]
kur = pd.to_numeric(kur, errors="coerce").dropna()                  # 1 sterlin kaç dolar, günlük
print(f"kur: {kur.index[0].date()} – {kur.index[-1].date()}, {len(kur)} gün")
_R = _referans(kur)

def kontrol_A(r, tablo, cv_r2, test_r2, naif_r2):
    if len(r) != len(_R["r"]) or not np.allclose(np.asarray(r), _R["r"].values):
        print("❌ r beklenenden farklı: r = 100 * np.log(kur).diff().dropna() (yüzde log getiri)."); return
    for c in ["v5", "v20", "v60", "y_v"]:
        if c not in tablo.columns:
            print(f"❌ tabloda {c} sütunu yok."); return
    if len(tablo) != len(_R["t"]) or not tablo.index.equals(_R["t"].index):
        print("❌ tablonun satırları beklenenden farklı: bütün sütunlar kurulduktan sonra dropna() uygula."); return
    for c in ["v5", "v20", "v60"]:
        if not np.allclose(tablo[c].values, _R["t"][c].values):
            print(f"❌ {c} beklenenden farklı: np.sqrt((r ** 2).rolling(w).mean()), yalnızca geçmişe bakan pencere (center=True yok)."); return
    if not np.allclose(tablo["y_v"].values, _R["t"]["y_v"].values):
        print("❌ y_v beklenenden farklı: önümüzdeki 20 günün volatilitesi, np.sqrt((r ** 2).rolling(20).mean().shift(-20))."); return
    if abs(float(cv_r2) - _R["cv"]) > 1e-6:
        if abs(float(cv_r2) - _R["cv_rand"]) < 1e-3:
            print("❌ cv_r2 rastgele bölmeyle (KFold, shuffle) hesaplanmış: zaman serisinde TimeSeriesSplit(5, gap=20) kullan."); return
        if abs(float(cv_r2) - _R["cv_nogap"]) < 1e-6 and abs(_R["cv_nogap"] - _R["cv"]) > 1e-6:
            print("❌ cv_r2'de boşluk yok: TimeSeriesSplit(n_splits=5, gap=20)."); return
        print("❌ cv_r2 beklenenden farklı: LinearRegression, X = v5, v20, v60; cv=TimeSeriesSplit(n_splits=5, gap=20), scoring='r2', ortalama."); return
    if abs(float(test_r2) - _R["test"]) > 1e-6:
        print("❌ test_r2 beklenenden farklı: model yalnızca 2015 öncesiyle kurulmalı, R² 2015 ve sonrasında hesaplanmalı."); return
    if abs(float(naif_r2) - _R["naive"]) > 1e-6:
        print("❌ naif_r2 beklenenden farklı: test döneminde r2_score(y, tablo['v60'])."); return
    print(f"✅ Tüm kontroller geçti (cv R² {_R['cv']:.3f}, test R² {_R['test']:.3f}, saf v60 {_R['naive']:.3f}). Tamamlama kodun: " + _k("TUw2LVZPTC00VFpD"))

def kontrol_B(tablo, q, asim_orani, asim_normal):
    if "y_r" not in tablo.columns or not np.allclose(tablo["y_r"].values, _R["r"].shift(-1).reindex(tablo.index).values, equal_nan=True):
        print("❌ tablo['y_r'] yarının getirisi olmalı: r.shift(-1)."); return
    if abs(float(q) - _R["q"]) > 1e-9:
        if abs(float(q) - _R["q_all"]) < 1e-9:
            print("❌ Çarpan bütün veriden (test dâhil) öğrenilmiş: yalnızca 2015 öncesindeki y_r / v60 oranının %1 yüzdeliğini kullan."); return
        print("❌ q beklenenden farklı: 2015 öncesinde (y_r / v60) oranının %1 yüzdeliği, np.quantile(..., 0.01)."); return
    if abs(float(asim_orani) - _R["exc"]) > 1e-9:
        print("❌ asim_orani beklenenden farklı: 2015 ve sonrasında (y_r < q * v60) ortalaması."); return
    if abs(float(asim_normal) - _R["exc_n"]) > 1e-9:
        print("❌ asim_normal beklenenden farklı: 2015 ve sonrasında (y_r < -2.326 * v60) ortalaması."); return
    print(f"✅ Tüm kontroller geçti (çarpan {_R['q']:.2f}; aşım oranı %{100 * _R['exc']:.2f}, normal çarpanla %{100 * _R['exc_n']:.2f}). Tamamlama kodun: " + _k("TUw2LVZBUi03QkhO"))

kur.tail(3)

## Görev A — Volatilite öngörüsü

1. Günlük log getiri: `r = 100 * np.log(kur).diff().dropna()`.
2. `tablo` adlı bir DataFrame: w = 5, 20, 60 için `v{w} = np.sqrt((r ** 2).rolling(w).mean())` (yalnızca geçmiş); hedef `y_v = np.sqrt((r ** 2).rolling(20).mean().shift(-20))` (önümüzdeki 20 gün); sonra `dropna()`.
3. `X = tablo[["v5", "v20", "v60"]]`, `y = tablo["y_v"]`. `LinearRegression` için `cross_val_score(..., cv=TimeSeriesSplit(n_splits=5, gap=20), scoring="r2")` ortalaması: `cv_r2`.
4. `egt = tablo.index < "2015-01-01"`, `tst = ~egt`. Modeli yalnızca `egt` ile kur; `tst` için R²: `test_r2`. Saf tahmin olarak `tablo.loc[tst, "v60"]`'ın R²'si: `naif_r2`.

**Denetim soruları:** Rastgele bölme (`train_test_split`, `KFold(shuffle=True)`) var mı? Pencereler geçmişe mi bakıyor? Hedef geleceğe mi?

In [ ]:
# Görev A: Gemini ile kodu burada üret, oku, sonra çalıştır.

### Kontrol A — çalıştır, değiştirme

In [ ]:
kontrol_A(r, tablo, cv_r2, test_r2, naif_r2)

## Görev B — Riske maruz değer (%1 VaR)

1. Tabloya yarının getirisini ekle: `tablo["y_r"] = r.shift(-1)`.
2. Çarpan: 2015 öncesinde `tablo["y_r"] / tablo["v60"]` oranının %1 yüzdeliği: `q = np.quantile(..., 0.01)` (eksik değerleri at).
3. Aşım: `tablo["y_r"] < q * tablo["v60"]`. 2015 ve sonrasındaki ortalaması: `asim_orani`.
4. Aynısını normal dağılım çarpanıyla (−2.326) yap: `asim_normal`. İkisini %1 hedefiyle karşılaştır ve aşımların yıllara göre dağılımına bak.

**Denetim soruları:** Çarpan yalnızca eğitim döneminden mi öğrenildi? Aşımlar hangi yıllarda yığılıyor?

In [ ]:
# Görev B: Gemini ile kodu burada üret, oku, sonra çalıştır.

### Kontrol B — çalıştır, değiştirme

In [ ]:
kontrol_B(tablo, q, asim_orani, asim_normal)

## Denetim notların

- Gemini zamana saygılı mı çalıştı? Neyi düzeltmen gerekti?
- Sterlin sonuçları euro/dolarla nasıl karşılaştırılıyor? (cv_r2, test_r2, naif_r2)
- VaR sonuçlarını bir risk komitesine nasıl anlatırsın? (aşım oranı, normal çarpanla fark, sınırlılıklar)